# Tool 7: PnL Simulator (Vectorized Backtester)

## Purpose
A fast, vectorized strategy backtester that runs in milliseconds inside a
notebook cell. Tests simple market-making and mean-reversion strategies
against historical orderbook data to estimate PnL.

## Why This Matters
Frankfurt Hedgehogs used "quick, vectorized backtests inside Jupyter
notebooks for rapid prototyping" before committing to full simulations.
This lets you test dozens of parameter combinations in seconds.

## What It Does
1. **Market Making Backtest**: For a given edge (distance from fair price),
   simulates placing passive bids/asks and checks whether historical
   trades would have filled them.
2. **Grid Search Heatmap**: Sweeps over parameter combinations (edge,
   max position) and plots PnL as a heatmap to find optimal settings.
3. **PnL Curve**: Cumulative PnL over time for the selected strategy.

## Important Caveats
- This is an **approximation**, not a full simulation. It assumes your
   orders would have been filled whenever a taker crosses your price.
- It does NOT model queue priority, partial fills, or bot interactions.
- Use this for **rapid prototyping only**. Validate final strategies on
   the official Prosperity backtester or website.
- Frankfurt Hedgehogs' key advice: "Never optimize purely for backtester
   score. Look for flat, stable regions of good performance."

## Dependencies
- `data_loader.py`, `matplotlib`, `pandas`, `numpy`

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

PRODUCT = "ASH_COATED_OSMIUM"  # or "INTARIAN_PEPPER_ROOT"
DAY = -1

# --- Market Making Strategy Parameters ---
# Edge: distance from Wall Mid at which to place passive orders.
# e.g., EDGE=2 means bid at (wall_mid - 2) and ask at (wall_mid + 2)
EDGE = 2

# Maximum position: flatten inventory when abs(position) exceeds this
MAX_POSITION = 50

# Flatten threshold: start flattening when abs(position) >= this
FLATTEN_THRESHOLD = 40

# --- Grid Search Parameters ---
# Set to True to run a sweep over edge and max_position values
RUN_GRID_SEARCH = True
EDGE_RANGE = range(1, 12)         # Edge values to test
MAX_POS_RANGE = range(10, 55, 5)  # Max position values to test

FIG_WIDTH = 18
FIG_HEIGHT = 6

In [ ]:
# ============================================================
# SETUP
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_prices, load_trades, compute_wall_mid, AVAILABLE_DAYS

prices = compute_wall_mid(load_prices(day=DAY, product=PRODUCT))
trades = load_trades(day=DAY, product=PRODUCT)

print(f"Loaded {len(prices)} snapshots, {len(trades)} trades for {PRODUCT} Day {DAY}")

In [ ]:
# ============================================================
# BACKTESTER ENGINE
# ============================================================

def backtest_market_making(prices_df, trades_df, edge, max_pos, flatten_thresh=None):
    """
    Vectorized backtest of a simple market-making strategy.

    Strategy logic (per timestep):
    1. If any existing ask <= wall_mid - edge: take it (buy below fair)
    2. If any existing bid >= wall_mid + edge: take it (sell above fair)
    3. Place passive bid at wall_mid - edge, ask at wall_mid + edge
    4. If abs(position) >= flatten_thresh: flatten toward 0 at wall_mid

    This is approximate — it checks if historical trades crossed our price
    levels, not whether we'd have queue priority.

    Parameters
    ----------
    prices_df : DataFrame with wall_mid, bid_price_1, ask_price_1
    trades_df : DataFrame with timestamp, price, quantity
    edge : float, distance from wall_mid for quoting
    max_pos : int, hard position limit
    flatten_thresh : int or None, start flattening at this position

    Returns
    -------
    dict with keys: total_pnl, pnl_series, position_series, n_trades
    """
    if flatten_thresh is None:
        flatten_thresh = max_pos

    wm = prices_df["wall_mid"].values
    best_bid = prices_df["bid_price_1"].values
    best_ask = prices_df["ask_price_1"].values
    timestamps = prices_df["timestamp"].values

    position = 0
    cash = 0.0
    n_trades = 0
    pnl_series = []
    position_series = []

    for i in range(len(wm)):
        fair = wm[i]
        if np.isnan(fair):
            pnl_series.append(cash + position * fair if not np.isnan(fair) else cash)
            position_series.append(position)
            continue

        # Take profitable asks (buy below fair - edge is not needed here,
        # we're checking if the ask is below our bid level)
        if not np.isnan(best_ask[i]) and best_ask[i] <= fair - edge and position < max_pos:
            buy_qty = min(1, max_pos - position)
            cash -= best_ask[i] * buy_qty
            position += buy_qty
            n_trades += 1

        # Take profitable bids (sell above fair + edge)
        if not np.isnan(best_bid[i]) and best_bid[i] >= fair + edge and position > -max_pos:
            sell_qty = min(1, max_pos + position)
            cash += best_bid[i] * sell_qty
            position -= sell_qty
            n_trades += 1

        # Passive fills: check if any trade crossed our quote levels
        ts_trades = trades_df[trades_df["timestamp"] == timestamps[i]]
        for _, trade in ts_trades.iterrows():
            # Someone bought at a price >= our ask -> we got filled on ask
            if trade["price"] >= fair + edge and position > -max_pos:
                sell_qty = min(int(trade["quantity"]), max_pos + position)
                if sell_qty > 0:
                    cash += (fair + edge) * sell_qty
                    position -= sell_qty
                    n_trades += 1
            # Someone sold at a price <= our bid -> we got filled on bid
            elif trade["price"] <= fair - edge and position < max_pos:
                buy_qty = min(int(trade["quantity"]), max_pos - position)
                if buy_qty > 0:
                    cash -= (fair - edge) * buy_qty
                    position += buy_qty
                    n_trades += 1

        # Flatten if over threshold
        if abs(position) >= flatten_thresh:
            cash += position * fair
            position = 0

        pnl_series.append(cash + position * fair)
        position_series.append(position)

    total_pnl = cash + position * wm[~np.isnan(wm)][-1] if len(wm[~np.isnan(wm)]) > 0 else cash

    return {
        "total_pnl": total_pnl,
        "pnl_series": np.array(pnl_series),
        "position_series": np.array(position_series),
        "n_trades": n_trades,
        "timestamps": timestamps,
    }

In [ ]:
# ============================================================
# PLOT 1: Single Strategy PnL Curve
# ============================================================

result = backtest_market_making(prices, trades, EDGE, MAX_POSITION, FLATTEN_THRESHOLD)

fig, axes = plt.subplots(2, 1, figsize=(FIG_WIDTH, FIG_HEIGHT * 1.5), sharex=True)

# PnL curve
axes[0].plot(result["timestamps"], result["pnl_series"], linewidth=1)
axes[0].set_ylabel("Cumulative PnL")
axes[0].set_title(
    f"Market Making Backtest — {PRODUCT} Day {DAY} "
    f"(edge={EDGE}, max_pos={MAX_POSITION}, flatten={FLATTEN_THRESHOLD})",
    fontweight="bold"
)
axes[0].grid(True, alpha=0.3)

# Position over time
axes[1].plot(result["timestamps"], result["position_series"], linewidth=0.8, color="tab:orange")
axes[1].axhline(MAX_POSITION, color="red", linestyle="--", alpha=0.5, label=f"±{MAX_POSITION} limit")
axes[1].axhline(-MAX_POSITION, color="red", linestyle="--", alpha=0.5)
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_ylabel("Position")
axes[1].set_xlabel("Timestamp")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Results:")
print(f"  Total PnL:  {result['total_pnl']:.0f}")
print(f"  Trades:     {result['n_trades']}")
print(f"  Max PnL:    {result['pnl_series'].max():.0f}")
print(f"  Min PnL:    {result['pnl_series'].min():.0f}")
print(f"  Max Drawdown: {(result['pnl_series'] - np.maximum.accumulate(result['pnl_series'])).min():.0f}")

In [ ]:
# ============================================================
# PLOT 2: Grid Search Heatmap
# ============================================================
# Sweeps over edge and max_position values. The heatmap shows total PnL
# for each combination. Look for FLAT, STABLE REGIONS of good performance
# rather than isolated peaks (which are likely overfit).

if RUN_GRID_SEARCH:
    edges = list(EDGE_RANGE)
    max_positions = list(MAX_POS_RANGE)
    results_grid = np.zeros((len(edges), len(max_positions)))

    print("Running grid search...")
    for i, e in enumerate(edges):
        for j, mp in enumerate(max_positions):
            r = backtest_market_making(prices, trades, e, mp, mp)
            results_grid[i, j] = r["total_pnl"]
        print(f"  edge={e}: done")

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(results_grid, aspect="auto", cmap="RdYlGn", origin="lower")
    ax.set_xticks(range(len(max_positions)))
    ax.set_xticklabels(max_positions)
    ax.set_yticks(range(len(edges)))
    ax.set_yticklabels(edges)
    ax.set_xlabel("Max Position")
    ax.set_ylabel("Edge")
    ax.set_title(f"PnL Grid Search — {PRODUCT} Day {DAY}", fontweight="bold")
    plt.colorbar(im, ax=ax, label="Total PnL")

    # Annotate cells with PnL values
    for i in range(len(edges)):
        for j in range(len(max_positions)):
            ax.text(j, i, f"{results_grid[i,j]:.0f}",
                    ha="center", va="center", fontsize=7,
                    color="white" if abs(results_grid[i,j]) > results_grid.max() * 0.6 else "black")

    plt.tight_layout()
    plt.show()

    best_idx = np.unravel_index(results_grid.argmax(), results_grid.shape)
    print(f"\nBest parameters: edge={edges[best_idx[0]]}, max_pos={max_positions[best_idx[1]]}")
    print(f"Best PnL: {results_grid.max():.0f}")
    print(f"\nREMINDER: Pick from a STABLE REGION, not the single best cell!")
else:
    print("Grid search disabled. Set RUN_GRID_SEARCH = True to run.")